In [1]:
# type: ignore

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

In [2]:
from langchain.tools import tool


@tool
def create_calendar_event(
    title: str,
    start_time: str,  # ISO format: "2024-01-15T14:00:00"
    end_time: str,  # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = "",
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = [],
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int,
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

In [3]:
from langchain.agents import create_agent


CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

In [4]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================

I'll help you schedule a team meeting for next Tuesday at 2pm. First, I need to check availability and then create the event.

Let me determine the exact date for next Tuesday and check if the time slot is available.
Tool Calls:
  get_available_time_slots (call_8514691384da45d39a7fe9a3)
 Call ID: call_8514691384da45d39a7fe9a3
  Args:
    attendees: ['user']
    date: 2023-10-31
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================

Great! I can see that 2:00 PM (14:00) is available next Tuesday. Now I'll create the calendar event for your team meeting.
Tool Calls:
  create_calendar_event (call_9763f6d1eaac46ae8a16fa2b)
 Call ID: call_9763f6d1eaac46ae8a16fa2b
  Args:
    title: Team Meeting
    start_time: 2023-10-31

In [5]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

In [6]:
query = "Send the R&D team a reminder about code review."

for step in email_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_1477e6fe0c2541f9acf4f463)
 Call ID: call_1477e6fe0c2541f9acf4f463
  Args:
    to: ['rd-team@example.com']
    subject: Reminder: Code Review
    body: Hi R&D Team,

This is a friendly reminder to please complete your pending code reviews at your earliest convenience. Timely reviews help us maintain code quality and keep our development pipeline moving smoothly.

Thank you!
================================= Tool Message =================================
Name: send_email

Email sent to rd-team@example.com - Subject: Reminder: Code Review
================================== Ai Message ==================================

I've successfully sent a reminder email to the R&D team (rd-team@example.com) with the subject "Reminder: Code Review." The message asked them to complete their pending code reviews promptly to maintain code quality and keep the development pipeline moving smoothl

In [7]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with R&D team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].text

In [8]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)

In [9]:
query = "Schedule a team standup for tomorrow at 9am"

for step in supervisor_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_34ad4593b7a442c4a0ac92f1)
 Call ID: call_34ad4593b7a442c4a0ac92f1
  Args:
    request: team standup tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

I'll help you schedule a team standup for tomorrow at 9am. First, I need to check if the time slot is available.

To do this properly, I need to know:
1. Who are the attendees for this standup?
2. How long will the standup last?

Once you provide this information, I can check availability and create the calendar event for you.
================================== Ai Message ==================================

To schedule your team standup for tomorrow at 9am, I need a bit more information:

1. **Who should be invited?** (team members, specific people, or just yourself?)
2. **How long should the meeting last?** (typically standups are 15-30 minutes, but I want

In [10]:
query = (
    "Schedule a meeting with the R&D team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new POCs."
)

for step in supervisor_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================

I'll help you schedule the meeting with the R&D team and send them an email reminder about reviewing the new POCs.

First, let me schedule the meeting:
Tool Calls:
  schedule_event (call_b7f3807bcbd64d208d2cdffc)
 Call ID: call_b7f3807bcbd64d208d2cdffc
  Args:
    request: meeting with R&D team next Tuesday at 2pm for 1 hour
  manage_email (call_a711ed8cd39f4d86801c883d)
 Call ID: call_a711ed8cd39f4d86801c883d
  Args:
    request: send the R&D team a reminder about reviewing the new POCs for their meeting next Tuesday at 2pm
================================= Tool Message =================================
Name: manage_email

The email reminder about reviewing the new POCs for the R&D team's meeting next Tuesday at 2 PM has been successfully sent to rd-team@company.com.
================================= Tool Message =================================
Name: schedule_event

Perfect! I've successfully scheduled

In [11]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"create_calendar_event": True},
            description_prefix="Calendar event pending approval",
        ),
    ],
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},
            description_prefix="Outbound email pending approval",
        ),
    ],
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),
)

In [ ]:
from langchain_core.runnables import RunnableConfig

query = (
    "Schedule a meeting with the R&D team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new POCs."
)

config = RunnableConfig({"configurable": {"thread_id": "1"}})

interrupts = []
for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")

================================== Ai Message ==================================

I'll help you schedule the meeting with the R&D team and send them an email reminder about reviewing the new POCs.

First, let me schedule the meeting:
Tool Calls:
  schedule_event (call_f64aa00354f34848a1dacaee)
 Call ID: call_f64aa00354f34848a1dacaee
  Args:
    request: meeting with R&D team next Tuesday at 2pm for 1 hour
  manage_email (call_8e224f3a37c94ddb8c5ab054)
 Call ID: call_8e224f3a37c94ddb8c5ab054
  Args:
    request: send the R&D team a reminder about reviewing the new POCs

INTERRUPTED: b14310d826e6ec65ddd4976c4d6d8184

INTERRUPTED: 47e5d4fb63fb160c8e9a3f5bc1925bf6


In [13]:
for interrupt_ in interrupts:
    for request in interrupt_.value["action_requests"]:
        print(f"INTERRUPTED: {interrupt_.id}")
        print(f"{request['description']}\n")

INTERRUPTED: b14310d826e6ec65ddd4976c4d6d8184
Outbound email pending approval

Tool: send_email
Args: {'to': ['rd-team@example.com'], 'subject': 'Reminder: Please Review New POCs', 'body': 'Hi R&D Team,\n\nThis is a friendly reminder to review the newly submitted proofs of concept (POCs). Your feedback is essential to move forward with the next steps.\n\nPlease aim to complete your reviews by the end of the week.\n\nThank you!'}

INTERRUPTED: 47e5d4fb63fb160c8e9a3f5bc1925bf6
Calendar event pending approval

Tool: create_calendar_event
Args: {'title': 'Meeting with R&D team', 'start_time': '2023-11-21T14:00:00', 'end_time': '2023-11-21T15:00:00', 'attendees': ['R&D team'], 'location': ''}



In [16]:
from langgraph.types import Command

resume = {}
for interrupt_ in interrupts:
    if interrupt_.id == "b14310d826e6ec65ddd4976c4d6d8184":
        # Edit email
        edited_action = interrupt_.value["action_requests"][0].copy()
        edited_action["args"]["subject"] = "Mockups reminder"
        resume[interrupt_.id] = {
            "decisions": [{"type": "edit", "edited_action": edited_action}]
        }
    else:
        resume[interrupt_.id] = {"decisions": [{"type": "approve"}]}

interrupts = []
for step in supervisor_agent.stream(
    Command(resume=resume),
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")

================================= Tool Message =================================
Name: schedule_event

Perfect! I've successfully scheduled your meeting with the R&D team for next Tuesday (November 21st) from 2:00 PM to 3:00 PM. The event has been created and confirmed in the calendar.
================================= Tool Message =================================
Name: manage_email

I've sent the following email to rd-team@example.com:

**Subject:** Mockups reminder  
**Body:**  
Hi R&D Team,

This is a friendly reminder to review the newly submitted proofs of concept (POCs). Your feedback is essential to move forward with the next steps.

Please aim to complete your reviews by the end of the week.

Thank you!
================================== Ai Message ==================================

Great! I've completed both tasks for you:

1. **Meeting Scheduled**: Your meeting with the R&D team has been successfully scheduled for next Tuesday (November 21st) from 2:00 PM to 3:00 PM. The ev